In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('.\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [3]:
tabla_departamento = pd.read_sql_table('departamento', mensajeria)
tabla_departamento.head(10)

,departamento_id,nombre
0,4,NARIÑO
1,3,CAUCA
2,2,CUNDINAMARCA
3,1,VALLE DEL CAUCA


In [4]:
tabla_ciudad = pd.read_sql_table('ciudad', mensajeria)
tabla_ciudad.head(10)

,ciudad_id,nombre,departamento_id
0,6,BUGA,1
1,5,BOGOTA,2
2,4,PASTO,4
3,3,POPAYAN,3
4,2,PALMIRA,1
5,1,CALI,1
6,7,TULUA,1
7,10,GUACARI,1
8,11,CANDELARIA,1
9,12,CARTAGO,1


In [5]:
dim_geografia = pd.merge(tabla_ciudad,tabla_departamento,left_on='departamento_id',right_on='departamento_id',how='left')
dim_geografia['key_geografia'] = range(1,len(dim_geografia)+1)
dim_geografia.drop(columns=['departamento_id'], inplace=True)
dim_geografia.rename(columns={'nombre_x':'ciudad','nombre_y':'departamento'},inplace=True)
dim_geografia.head(20)

,ciudad_id,ciudad,departamento,key_geografia
0,6,BUGA,VALLE DEL CAUCA,1
1,5,BOGOTA,CUNDINAMARCA,2
2,4,PASTO,NARIÑO,3
3,3,POPAYAN,CAUCA,4
4,2,PALMIRA,VALLE DEL CAUCA,5
5,1,CALI,VALLE DEL CAUCA,6
6,7,TULUA,VALLE DEL CAUCA,7
7,10,GUACARI,VALLE DEL CAUCA,8
8,11,CANDELARIA,VALLE DEL CAUCA,9
9,12,CARTAGO,VALLE DEL CAUCA,10


In [6]:
dim_geografia.to_sql('dim_geografia', etl_conn, if_exists='replace', index=False)

19

In [7]:
dim_geografia.to_sql('dim_geografia', etl_conn, if_exists='replace', index=False)

19